# Voxel-wise kinetic parameters of [177Lu]Lu-PSMA therapy from the PSMA PET/CT: checks, benchmark, training and evaluation

The code lives in the `lib/` package on Drive (`<DRIVE_ROOT>/deep_learning/`, `DRIVE_ROOT` set in the configuration cell); this notebook only contains configuration and execution. The voxel targets are rebuilt by `src/net_data.py` (`<DRIVE_ROOT>/src/`), which is imported, not copied. Doses are not computed here: the network outputs kinetic parameters, the doses are computed locally with `src/cohort_doses.py`.

Step 0 checks the data and the target rebuild on the GPU, step 1 measures the speed of the training step and estimates the cost of the training runs. The resume test (section 7) and the training (section 8) are switched off by their flags. Section 9 evaluates the best weights of a scenario on the validation phantoms; section 10 holds the test evaluation and section 12 the weight-space SPECT update (M4) on the test split, both switched off until the frozen test run (`test_run.py`, see section 10). With the defaults (`SCENARIO = "A"`, every `RUN_*` flag `False`) Run all only imports the code and reads two small tables.

| Module | Content |
|---|---|
| `config.py` | `SamplingConfig`, `AugmentConfig`, `LossConfig`, `TrainConfig`, the output heads (`HEADS`) |
| `data.py` | `check_parquet`, `GpuCohort` (all phantoms and the lookup tables of every draw on the GPU), `sample_batch` (patch sampling, augmentation, target rebuild on the GPU), `check_rebuild` |
| `model.py` | `KineticSwinUNETR`, self-supervised weights adapted to 2 channels, `set_checkpointing`, output biases from the population prior |
| `losses.py` | `voxel_targets`, `kinetic_loss` (masked heteroscedastic NLL balanced by structure, consistency term) |
| `evaluate.py` | `predict_phantom` (MONAI sliding windows), `validate` (NLL over the draws of each validation phantom), `prior_output` (the population prior as a voxel predictor), `load_best` |
| `structures.py` | structure masks (lesions dilated by one voxel), structure quantities of the rebuilt targets and of the network output, predictive SD by sampling |
| `calibration.py` | `evaluate_cohort` (one pass over the phantoms), `VoxelStats` (voxel NLL and calibration in the loss metric), variance scales and their cross-validation, `compare_methods` (through `net_data.evaluate`), `voxel_report`, `structure_report`, `build_test_cohort` |
| `scoring.py`, `gating.py`, `diagnostics.py` | no torch, also used locally by `diagnose.py`: table functions (scales, comparison), the gated prior, the diagnostics from the saved maps |
| `kinetics.py`, `spect.py` | differentiable kinetics of the voxel parameters (activity at any time, TIA); virtual SPECT: forward model (resolution, calibration) and noise |
| `update.py`, `tto.py`, `pbpk_torch.py` | single-SPECT update: M1 and M2 (structure level), M3 and M4 (test-time optimisation of the maps and of the weights), M5 (differentiable PBPK). Only M4 runs here (section 11); the others run locally on CPU (`spect_update*.py`) |
| `figures.py` | reliability curves, z histograms, predicted against true TIAC, coverage per structure |
| `train.py` | `micro_step`, `optimizer_step`, `train` (gradient accumulation, resume, timed checkpoints, early stopping), `resume_test` |
| `benchmark.py` | `run_benchmark`, `time_validation`, `estimate_cost` |
| `utils.py` | device, bf16 autocast, scheduler, `BackgroundSaver`, `GpuMonitor`, `stop_runtime` |

The test split is never read by sections 0-9: `data.py` refuses every file whose name contains `_test` or starts with `test_`. Only sections 10 and 12, behind `RUN_TEST_EVALUATION` and `RUN_SPECT_UPDATE_TEST`, open it, through `calibration.build_test_cohort`.

### 0. Environment
Installs MONAI and mounts Google Drive. Outside Colab the cell does nothing and the paths of section 1 point to the local repository.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q monai==1.6.0 einops
    from google.colab import drive
    drive.mount('/content/drive')

### 1. Package import
`CODE_DIR` is the folder containing `lib/`, `SRC_DIR` the one containing `net_data.py`. The cell removes the package from `sys.modules` before importing it: after editing a `.py` file on Drive, re-running this cell reloads it without restarting the runtime.

In [ ]:
import json
import os
import time
import warnings

DRIVE_ROOT = "/content/drive/MyDrive/lu177"
ROOT = DRIVE_ROOT if IN_COLAB else os.path.abspath("..")
CODE_DIR = f"{ROOT}/deep_learning"
SRC_DIR = f"{ROOT}/src"
if not os.path.isfile(os.path.join(CODE_DIR, "lib", "__init__.py")):
    raise FileNotFoundError(f"Cannot find {CODE_DIR}/lib/__init__.py: check CODE_DIR")
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

for module in [m for m in sys.modules if m == "lib" or m.startswith("lib.")]:
    del sys.modules[module]

import numpy as np
import pandas as pd
import torch

from lib.benchmark import estimate_cost, run_benchmark, time_validation
from lib.calibration import (CALIBRATED, INDEPENDENT, RAW, build_test_cohort, evaluate_cohort, file_md5, maps_size_gb,
                             side_by_side, structure_report, test_file_names, voxel_report)
from lib.config import HEADS, AugmentConfig, LossConfig, SamplingConfig, TrainConfig
from lib.evaluate import load_best, predict_phantom
from lib.spect import SpectConfig
from lib.scenario import CEILING, scenario_file
from lib.scoring import with_sd
from lib.update import case_data, gated_table, mismatch
from lib.tto import weight_update_rows
from lib.data import GpuCohort, check_parquet, check_rebuild, import_net_data, shard_names
from lib import frozen
from lib.gating import apply_choice
from lib.model import build_model, set_checkpointing
from lib.train import resume_test, train
from lib.utils import (clear_vram, copy_files_to_local, get_device, gpu_name, provenance, save_table, set_seed,
                       stop_runtime)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 250, "display.max_columns", 40, "display.max_rows", 200)


def require(*names, hint):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(f"{', '.join(missing)} not defined: {hint}")


nd = import_net_data(SRC_DIR)
device = get_device()
STAMP = provenance(CODE_DIR, SRC_DIR)
print(f"Code commit {STAMP['code_commit']} | net_data.py md5 {STAMP['net_data_md5']}")

### 2. Configuration
All paths and flags in one place; the long stages only run if their flag is `True`.

- **Defaults are safe**: `SCENARIO = "A"` and every `RUN_*` flag `False`, so Run all on Colab copies nothing and runs nothing expensive; set only the flags of the stage you want. `LU177_NETDATA` and `LU177_RUNS` (environment variables, unset on Colab) redirect the data and the outputs: the dry run of `test_run.py` uses them to execute sections 10 and 12 on a stand-in test split.
- **Flags**: `RUN_BENCHMARK`, `RUN_RESUME_TEST` and `RUN_TRAINING` need the training phantoms; if none of them is set, sections 4 and 5 are skipped and only the validation shard is copied. `RUN_EVALUATION` (section 9) builds its own cohort with the validation phantoms only. `RUN_TEST_EVALUATION` (section 10) and `RUN_SPECT_UPDATE_TEST` (section 12) stay `False` until the frozen test run; `TEST_CU_LIMIT` caps the estimated cost of section 12. `SAVE_VAL_MAPS` also writes the network maps of the validation phantoms, which the local diagnostics (`diagnose.py`) read. `RUN_TRAIN_CHECK` (section 9.5) evaluates `TRAIN_CHECK_PHANTOMS` training phantoms with the same code and needs the training shards on the local disk.
- **Scenario**: `SCENARIO` is the label of `params.parquet` (`A`, `B+0.6`, ...). The file names of a run carry it (`best_B+0.6.pth`, `runs/eval_B+0.6/`), so a B run never writes on the files of A; training A again while `best_A.pth` exists is refused unless `ALLOW_RETRAIN_A`. The tables that depend on the scenario (`population_prior_<scenario>.parquet`, `baseline_population_<scenario>.parquet`, `baseline_same_phantom_draws_<scenario>.parquet`) are built locally by `scenario_baselines.py` and must be in `netdata/`; without them the cells stop (no silent fall-back on the tables of A).
- **Budget**: `TRAIN_CU_LIMIT` is a hard stop of the training in compute units, counted over all the sessions of the run (training and validations; the time is kept in the checkpoint): when it is reached a checkpoint is written and the training returns, resumable. It leaves room for the set-up of the session and for the evaluation under 8 CU.
- **Local data**: the training and validation shards are copied once per session to the Colab disk, then loaded whole into GPU memory (section 4). There is no CPU dataloader.
- **Patches**: 96³ and 128³, plus 128 × 96 × 160. The phantoms measure 104-135 × 72-104 × 144-394 voxels (4 mm): they are narrower than 96 voxels along y in most cases, so a cube spends part of its volume on padding; the third shape follows the body.
- **Sampling**: 70% of the patches are centred (with a jitter of a quarter of the patch) on a voxel of an organ or lesion, the class drawn uniformly among those present; the others anywhere in the volume. Each sample picks one of the 10 draws of the phantom at random.
- **Augmentation**: left-right flip only (the other axes are not anatomical), CT shift of ±20 HU, Gaussian noise on the PET. The PET is not rescaled (`pet_scale=0.0`): scaling it without scaling the A0 target would teach the network that the PET amplitude matters less than it does.
- **Provenance**: every table saved in `runs/` carries two columns, `code_commit` and `net_data_md5`, and every checkpoint a `provenance` entry with the same two values. Drive has no git repository, so the commit is read from the `VERSION` file uploaded with the code (written locally by `lib.utils.write_version` after each commit of `deep_learning/`); it is marked `+modified` if the files of `lib/` no longer match the digest stored with it. The notebook itself is not part of that digest.
- **Cost**: `CU_PER_HOUR` is the rate of the runtime in compute units per hour, to be read from the Colab resources panel (it changes over time): with `None` only the hours are given. `PASSES` is the assumed number of passes over the training voxels needed by scenario A, `FINETUNE_FRACTION` the share of it assumed for each B scenario. Both are assumptions, not measurements.

In [ ]:
RUN_BENCHMARK = False
RUN_RESUME_TEST = False
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_TEST_EVALUATION = False
SAVE_VAL_MAPS = False
RUN_TRAIN_CHECK = False
TRAIN_CHECK_PHANTOMS = 12
RUN_SPECT_UPDATE = False
RUN_SPECT_UPDATE_TEST = False
SPECT_STAGE = "reporting"
SPECT_TAG = "tuning2"
SPECT_CU_LIMIT = 1.5
TEST_CU_LIMIT = 3.0
NEED_TRAINING_COHORT = RUN_BENCHMARK or RUN_RESUME_TEST or RUN_TRAINING
NEED_VAL_SHARDS = NEED_TRAINING_COHORT or RUN_EVALUATION or RUN_TRAIN_CHECK or RUN_SPECT_UPDATE
STOP_RUNTIME_AT_END = True
SEED = 20261113
set_seed(SEED)

NETDATA = os.environ.get("LU177_NETDATA") or (f"{ROOT}/netdata" if IN_COLAB else f"{ROOT}/data/processed/psma/netdata")
RUN_DIR = os.environ.get("LU177_RUNS") or f"{CODE_DIR}/runs"
LOCAL_DATA = "/content/local_data"
SHARD_DIR = (copy_files_to_local(f"{NETDATA}/shards", shard_names(NETDATA, ("train", "val") if NEED_TRAINING_COHORT or RUN_TRAIN_CHECK
                                              else ("val",)),
                                  f"{LOCAL_DATA}/shards")
             if IN_COLAB and NEED_VAL_SHARDS else f"{NETDATA}/shards")
os.makedirs(RUN_DIR, exist_ok=True)

SCENARIO = "A"
FEATURE_SIZE = 48
PATCHES = [(96, 96, 96), (128, 128, 128), (128, 96, 160)]
BENCHMARK_SECONDS = 75
VALIDATION_PATCH = (128, 96, 160)

CU_PER_HOUR = 1.94
PASSES = (300, 600)
FINETUNE_FRACTION = 0.25
STEPS_A = 21300
L4_STEPS_PER_S = 0.443
TRAIN_CU_LIMIT = 7.3
ALLOW_RETRAIN_A = False
TRAINING = {"A": dict(total_steps=STEPS_A, warmup_steps=500, lr=1e-4, val_every=500, patience=10, init_weights=None),
            "B": dict(total_steps=int(round(STEPS_A * FINETUNE_FRACTION)), warmup_steps=200, lr=3e-5, val_every=500, patience=6,
                      init_weights=f"{RUN_DIR}/best_A.pth")}

sampling_cfg = SamplingConfig(p_foreground=0.7, jitter=0.25,
                              augment=AugmentConfig(flip_axes=(0,), pet_scale=0.0, pet_noise=0.02, ct_shift_hu=20.0))
loss_cfg = LossConfig(alpha=0.5, consistency=0.1)

EVAL_DIR = f"{RUN_DIR}/eval_{SCENARIO}"
EVAL_SAMPLES = 64
EVAL_SEED = 20261114
EVAL_FOLDS = 5

### 3. Step 0a: parquet files
Skipped unless a stage is switched on. Every parquet file of `netdata/` except the test ones is read with the pandas and pyarrow of the runtime. If one fails, the cell installs the versions the files were written with (pandas 3.1.0, pyarrow 26.0.0) and asks for a restart of the runtime.

In [ ]:
ANY_STAGE = NEED_VAL_SHARDS or RUN_TEST_EVALUATION or RUN_SPECT_UPDATE_TEST
parquet_table = check_parquet(NETDATA) if ANY_STAGE else None
if ANY_STAGE:
    print(parquet_table.to_string(index=False))
if ANY_STAGE and not parquet_table.ok.all():
    !pip install -q pandas==3.1.0 pyarrow==26.0.0
    raise RuntimeError("Some parquet files could not be read: pandas 3.1.0 and pyarrow 26.0.0 installed, "
                       "restart the runtime and run again")

### 4. Data in GPU memory and step 0b: target rebuild on the GPU
`GpuCohort` loads the training and validation phantoms on the GPU as stored (PET, CT and weights in float16, compartment in uint8, lesion ids in int16) and builds, with `net_data.parameter_arrays`, the lookup tables of the 10 draws of every phantom for the chosen scenario (main arm).

`check_rebuild` compares `net_data.rebuild` with torch tensors on the GPU against NumPy float64 on two training phantoms (the one with most lesions and one from another scanner), draw 0: in float64, where the two should agree to rounding, and in float32, the precision used in training. For every returned field: the largest voxel-wise relative difference, the largest absolute difference over the largest value, and whether the masks (and the pattern of infinite $\lambda_u$) are identical.

In [ ]:
if NEED_TRAINING_COHORT:
    t0 = time.perf_counter()
    cohort = GpuCohort(nd, NETDATA, SHARD_DIR, device, scenario=SCENARIO)
    counts = cohort.phantoms.net_split.value_counts()
    print(f"{counts['train']} training and {counts['val']} validation phantoms on {device} in {time.perf_counter() - t0:.0f} s, "
          f"{cohort.memory_gb():.2f} GB | scenario {cohort.scenario}, {cohort.n_draws[0]} draws per phantom")

    rebuild_table = check_rebuild(cohort, NETDATA, SHARD_DIR)
    print(rebuild_table.groupby(["path", "field"], sort=False)
          .agg(max_relative=("max_relative", "max"), max_abs_over_max=("max_abs_over_max", "max"),
               pattern_equal=("pattern_equal", "all")).to_string())
    save_table(rebuild_table, f"{RUN_DIR}/rebuild_check_{gpu_name().replace(' ', '_')}.csv", STAMP)
    if not rebuild_table.pattern_equal.all():
        raise RuntimeError("The masks of the torch rebuild differ from NumPy")

### 5. Model
MONAI SwinUNETR (Hatamizadeh et al. 2022, `hatamizadeh2022`) with `feature_size` 48 and the MONAI self-supervised weights of the Swin encoder (Tang et al., CVPR 2022, `tang2022`; keys in `references.bib`). Those weights have one input channel (CT): the patch embedding is repeated on the two channels (PET, CT) and halved, so the first layer starts as the pretrained one applied to their mean and is then trained.

The network has 10 output channels, a mean and a log-variance for each of 5 heads, following the target kinds of the `netdata` README:

| Head | Scale | Voxels in the loss |
|---|---|---|
| $\log A_0$ per mL | log | body, where the source weight is positive |
| $\log \lambda_{bio}$ | log, floored at $10^{-5}$ /h | body |
| early TIA share (early TIA / TIA) | linear | body |
| $\log \lambda_u$ | log | `curve` structures (salivary glands, lesions) with a finite $\lambda_u$ (`lu_target`) |
| $\log$ TIAC per mL | log | `tia` structures (bladder, GI tract, blood pool, spine, rest of body) |

In the `tia` structures $A_0$ and $\lambda_{bio}$ are still trained: they describe the late part only, as in the README. The early share is constant inside a source, because early and late TIA are spread with the same weights. The output biases start at the population prior (`population_prior.parquet`, training part only): mean over the structures for the means, log of the pooled variance for the log-variances.

**Loss** (`kinetic_loss`): Gaussian negative log-likelihood per voxel and head, in float32, only on the voxels of the table. Each voxel has the weight $\alpha\, w_{structure} + (1-\alpha)/N$: the second term is uniform over the $N$ voxels of the head in the batch; the first gives the same total weight to every class present (the 9 compartments and the lesions as one class), split equally among the lesions and then among the voxels of each structure, so a 30-voxel lesion counts as much as any other lesion and the lesions together as much as the liver. The consistency term is the variance of the predicted mean $\log \lambda_{bio}$ inside each compartment and each lesion, averaged with the same class balance.

In [ ]:
prior = pd.read_parquet(f"{NETDATA}/population_prior.parquet")
eval_prior = pd.read_parquet(scenario_file(NETDATA, "population_prior", SCENARIO))
if NEED_TRAINING_COHORT:
    model = build_model(device, prior, feature_size=FEATURE_SIZE)
    print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parameters | heads: {', '.join(HEADS)}")

### 6. Step 1: benchmark and cost estimate
For each patch shape, with and without activation checkpointing of the Swin blocks (`use_checkpoint`): the largest batch that completes two training steps without running out of memory (candidates 1, 2, 3, 4, 6, 8, 12, 16), then `BENCHMARK_SECONDS` of real training steps (sampling, augmentation, target rebuild, forward in bf16, loss, backward, AdamW). Columns: iterations and patches per second, millions of voxels per second, share of the time spent on sampling and targets (`data_share`), peak memory allocated and reserved by PyTorch, GPU utilisation and memory from `nvidia-smi` sampled every second. The loss at the start and at the end of each run is only a sanity check. `nonfinite_steps` counts the steps whose loss or gradient norm was not finite under bf16 (the update is skipped for those): it should be 0; if it is not, the normalisation layers have to run in float32 before any training. The benchmark changes the model weights: sections 7 and 8 build a new model.

Then one full validation pass is timed (sliding windows on the 27 validation phantoms, loss on their 10 draws).

**Cost estimate.** The benchmark measures speed, not convergence. The hours are speed × an assumed budget: one pass = the number of patches needed to see every training voxel once (only the part of a patch inside the phantom counts, so padded cubes are not favoured), scenario A at each value of `PASSES`, a validation every 1000 iterations, each B scenario at `FINETUNE_FRACTION` of A starting from the A weights. Only the first hours of the real A run can replace these assumptions. Tables are saved in `runs/` with the GPU name, so the L4 and A100 runs can be compared.

In [ ]:
if RUN_BENCHMARK:
    tag = gpu_name().replace(" ", "_")
    t0 = time.perf_counter()
    benchmark_table = run_benchmark(model, cohort, PATCHES, sampling_cfg, loss_cfg, seconds=BENCHMARK_SECONDS)
    save_table(benchmark_table, f"{RUN_DIR}/benchmark_{tag}.csv", STAMP)
    print("\n" + benchmark_table.round(3).to_string(index=False))

    set_checkpointing(model, True)
    validation_seconds, _ = time_validation(model, cohort, VALIDATION_PATCH, loss_cfg)
    print(f"\nOne validation pass ({'x'.join(map(str, VALIDATION_PATCH))} windows): {validation_seconds:.0f} s")

    cost_table = estimate_cost(benchmark_table, cohort, validation_seconds, val_every=1000, passes=PASSES,
                               finetune_fraction=FINETUNE_FRACTION, cu_per_hour=CU_PER_HOUR)
    save_table(cost_table, f"{RUN_DIR}/cost_estimate_{tag}.csv", STAMP)
    print("\n" + cost_table.round(2).to_string(index=False))
    print(f"\nBenchmark finished in {(time.perf_counter() - t0) / 60:.1f} min; tables in {RUN_DIR}")
    clear_vram()

### 7. Resume test (off by default)
A short run of 6 optimiser steps (18 micro-batches, two validations) is done twice from the same initial model: once without interruption, once stopped after 4 micro-batches, that is inside an accumulation cycle, and resumed from the checkpoint. The table gives the largest difference between the final weights of the two runs and between their logs (they should agree to rounding: the checkpoint restores model, optimiser, scheduler, optimiser step, position in the accumulation cycle with the gradients accumulated so far, sampling RNG, best NLL and early-stopping counter). Then the last checkpoint of the resumed run is truncated on purpose, to check that loading falls back to the previous one (`resume_<scenario>.pth.prev`). The checkpoints of the test are deleted at the end; logs and table stay in `runs/`.

In [ ]:
if RUN_RESUME_TEST:
    def make_model():
        set_seed(SEED)
        return build_model(device, prior, feature_size=FEATURE_SIZE)

    test_cfg = TrainConfig(scenario=SCENARIO, total_steps=6, warmup_steps=2, val_every=3, log_every=1)
    resume_table = resume_test(make_model, cohort, test_cfg, sampling_cfg, loss_cfg, RUN_DIR, STAMP)
    save_table(resume_table, f"{RUN_DIR}/resume_test_{gpu_name().replace(' ', '_')}.csv", STAMP)
    print("\n" + resume_table.T.to_string(header=False))
    clear_vram()

### 8. Training (switched off)
Choices taken from the L4 benchmark: patch 128 × 96 × 160, no activation checkpointing, batch 1 with the gradients accumulated over 3 micro-batches (effective batch 3). The learning-rate schedule, the log, the validations and the budget count optimiser steps. Scenario A: 300 passes over the training voxels = 21 300 optimiser steps, validation every 500, early stopping after 10 validations without improvement. The gradient norm is clipped at 100: the median norm in the benchmark is 25-47, so this only cuts spikes (a clip at 1 would act on every step). The B scenarios start from the best A weights (`runs/best_A.pth`), same network and code, with `FINETUNE_FRACTION` of the budget (5325 optimiser steps), a warm-up of 200 steps, a peak learning rate of 3e-5 (scenario A, at 1e-4, became overconfident on validation while the training NLL kept falling) and early stopping after 6 validations without improvement (`TRAINING` in section 2; first choices, not tuned).

AdamW (the pretrained transformer at 0.2 times the learning rate of the rest), linear warm-up over 500 steps and cosine decay, bf16 autocast, TF32. The best model goes to `runs/best_<scenario>.pth`. `runs/train_log_<scenario>.csv` gets a row every `log_every` optimiser steps (mean training NLL since the previous row, gradient norm, skipped steps, learning rate, speed, memory, GPU utilisation), `runs/val_log_<scenario>.csv` a row per validation, and `runs/curves_<scenario>.png` shows the two NLL curves; the three files are updated at every log line and validation. The logs are stored in the checkpoint and the files rewritten from it, so a resumed run leaves no duplicated rows. A checkpoint (`runs/resume_<scenario>.pth`) is written in the background every `checkpoint_minutes`, at each validation and at the end: to a temporary file first, then renamed, and the file it replaces is kept as `.prev`, which is used if the last one cannot be read. The run resumes from it automatically, also from inside an accumulation cycle; to restart from scratch, delete both files. A step whose accumulated gradient norm is not finite is skipped and counted (`skipped` in the log).

In [ ]:
if RUN_TRAINING:
    settings = TRAINING["A" if SCENARIO == "A" else "B"]
    if SCENARIO == "A" and os.path.exists(f"{RUN_DIR}/best_A.pth") and not ALLOW_RETRAIN_A:
        raise RuntimeError("runs/best_A.pth exists: training scenario A again would overwrite its files (ALLOW_RETRAIN_A)")
    if settings["init_weights"] and not os.path.isfile(settings["init_weights"]):
        raise FileNotFoundError(f"{settings['init_weights']} not found: the fine-tuning starts from the best A weights")
    set_seed(SEED)
    model = build_model(device, prior, feature_size=FEATURE_SIZE)
    train_cfg = TrainConfig(scenario=SCENARIO, patch=(128, 96, 160), batch=1, accum_steps=3, checkpointing=False,
                            lr=settings["lr"], total_steps=settings["total_steps"], warmup_steps=settings["warmup_steps"],
                            val_every=settings["val_every"], patience=settings["patience"], grad_clip=100.0,
                            checkpoint_minutes=15, init_weights=settings["init_weights"])
    max_hours = TRAIN_CU_LIMIT / CU_PER_HOUR if CU_PER_HOUR and TRAIN_CU_LIMIT else None
    expected = train_cfg.total_steps / L4_STEPS_PER_S / 3600
    print(f"Scenario {SCENARIO}: {train_cfg.total_steps} optimiser steps, warm-up {train_cfg.warmup_steps}, peak lr {train_cfg.lr:g}, "
          f"validation every {train_cfg.val_every}, patience {train_cfg.patience}, from {train_cfg.init_weights} | "
          f"{expected:.2f} h of steps at the L4 speed of scenario A" + (f" = {expected * CU_PER_HOUR:.1f} CU" if CU_PER_HOUR else "")
          + (f" | hard stop at {max_hours:.2f} h = {TRAIN_CU_LIMIT} CU over all the sessions of the run" if max_hours else ""))
    best_nll = train(model, cohort, train_cfg, sampling_cfg, loss_cfg, RUN_DIR, stamp=STAMP, max_hours=max_hours)
    print(f"Best validation NLL: {best_nll:.4f}")


### 9. Evaluation on validation
The best weights of the scenario (`runs/best_<scenario>.pth`) are evaluated on the 27 validation phantoms and their 10 draws. Nothing here reads the test split, and nothing here selects a checkpoint: the scales fitted below are diagnostics and a post-hoc calibration, the early stopping stays on the raw validation NLL.

The training cohort is not needed: the cell frees it if it is in memory and builds a cohort with the validation phantoms only (0.5 GB). One pass over the phantoms (`evaluate_cohort`) does everything the next cells report: the sliding-window prediction with the settings of the training validation (`VALIDATION_PATCH`, overlap 0.25), the voxel statistics of the network and of the prior predictor on every draw, the structure predictions with their sampled uncertainty, and the structure summaries of the rebuilt targets. Tables and figures go to `runs/eval_<scenario>/` with the provenance stamp, to which the step and the validation NLL of the checkpoint are added.

`SAVE_VAL_MAPS` also writes the mean and log-variance maps of the five heads as float16 (`maps/<study_id>.npz`), for the local dose computation with `cohort_doses.py` and the later SPECT update; the size is printed before writing. float16 resolves 0.008 on values between 8 and 16 in magnitude (log A0 per mL), that is below 1% on A0.

In [ ]:
if RUN_EVALUATION:
    for name in ("cohort", "model"):
        globals().pop(name, None)
    clear_vram()
    os.makedirs(EVAL_DIR, exist_ok=True)
    val_cohort = GpuCohort(nd, NETDATA, SHARD_DIR, device, scenario=SCENARIO, splits=("val",))
    val_idx = val_cohort.split_indices("val")
    eval_model = build_model(device, feature_size=FEATURE_SIZE, load_ssl=False, use_checkpoint=False)
    best_step, best_nll, written_by = load_best(eval_model, f"{RUN_DIR}/best_{SCENARIO}.pth", device)
    EVAL_STAMP = {**STAMP, "checkpoint_step": best_step, "checkpoint_val_nll": best_nll}
    print(f"{len(val_idx)} validation phantoms, {val_cohort.memory_gb():.2f} GB | best_{SCENARIO}.pth: optimiser step "
          f"{best_step}, validation NLL {best_nll:.4f}, written by {written_by}")
    if SAVE_VAL_MAPS:
        print(f"Maps of the validation phantoms: {maps_size_gb(val_cohort, val_idx):.2f} GB in {EVAL_DIR}/maps")

    t0 = time.perf_counter()
    results = evaluate_cohort(lambda i: predict_phantom(eval_model, val_cohort, i, VALIDATION_PATCH, 2, 0.25),
                              val_cohort, val_idx, loss_cfg, eval_prior, k_samples=EVAL_SAMPLES, seed=EVAL_SEED,
                              maps_dir=f"{EVAL_DIR}/maps" if SAVE_VAL_MAPS else None)
    print(f"Evaluation pass: {(time.perf_counter() - t0) / 60:.1f} min")

#### 9.1 Voxel level: network against the population prior, and calibration
**Same metric as the training.** The prior predictor outputs, for every voxel, the mean and log(SD²) of its structure in `population_prior.parquet` for every head (the lesion row for lesion voxels), and goes through `kinetic_loss` with the same `LossConfig`, phantoms and draws as the network. `voxel_loss_check` verifies that the sums used here reproduce `kinetic_loss`, i.e. what `validate` returns. In `voxel_nll`, the row `all` of a head is its validation NLL (the classes add up to it through `weight_share`); the row of a class is the weighted mean NLL of its voxels. `network_minus_prior` > 0 means the network is worse than a constant.

What to expect in scenario A: the washout is drawn independently of the PET, so on `log_lambda_bio` the network cannot beat the prior and should match it; a clearly worse NLL with `mean_z2` well above 1 is overconfidence, a worse NLL with `mean_z2` near 1 is a wrong mean. On `log_a0` and `log_tiac` the prior is a constant per structure with the between-patient SD: it ignores how the activity is distributed inside the structure, so the network should be far better there.

**Calibration.** z = (target − mean) / SD on the masked voxels, with the weights of the loss (α × balanced by structure + (1 − α) × uniform). A calibrated head has weighted mean z = 0, mean z² = 1, and 68.3% and 95% of the weight inside |z| < 1 and |z| < 1.96. The reliability curve gives the empirical coverage at 10 nominal levels (coverages are read from a histogram of |z| with bins of 0.005, linearly interpolated). The NLL-optimal factor on the SD of a head is s = sqrt(weighted mean z²) (`head_scale`; `own_scale` is the same for one class); `nll_scaled` is the validation NLL after multiplying every SD of the head by s. This factor is fitted on the same data it is assessed on and is a diagnostic: it is not used to select checkpoints.

In [ ]:
if RUN_EVALUATION:
    voxel_nll, voxel_calibration, loss_check = voxel_report(results, EVAL_DIR, SCENARIO, EVAL_STAMP)
    print("kinetic_loss against the sums of the report:\n" + loss_check.round(4).to_string(index=False))
    print("\nNetwork against prior, per head:\n" + voxel_nll[voxel_nll.structure == "all"].drop(columns="structure").round(3).to_string(index=False))
    print("\nNetwork minus prior NLL by structure class (weighted mean over the voxels of the class):\n"
          + voxel_nll[voxel_nll.structure != "all"].pivot(index="structure", columns="head", values="network_minus_prior")[list(HEADS)].round(3).to_string())
    shown = ["head", "structure", "mean_z", "mean_z2", "own_scale", "coverage_68", "coverage_95", "nll", "nll_scaled",
             "coverage_68_scaled", "coverage_95_scaled"]
    print("\nCalibration:\n" + voxel_calibration[shown].round(3).to_string(index=False))

#### 9.2 Structure level: predictions keyed like the baselines
For every phantom the voxel maps are turned into one row per structure with `<q>_mean` and `<q>_sd` for the quantities of `net_data.EVAL_QUANTITIES`, in the units of `params.parquet` (fraction of the injected activity, 1/h, h). In scenario A the prediction does not depend on the draw, so it is repeated for the 10 virtual patients of the phantom. There is no `<q>_n` column: `net_data.evaluate` scores the network as Gaussian.

- **Aggregation.** A0 = sum of the voxel A0 (per-mL head × voxel volume); λ_u = A0-weighted mean over the voxels of `curve` structures, the only ones where its head is trained; λ_bio = median of the voxel values in a compartment, A0-weighted median on a lesion mask; late TIA = sum of A0 / λ_w; TIA of a voxel = the TIAC head on `tia` structures, late / (1 − early share) elsewhere; early share = summed early TIA / summed TIA. The early share is capped at 0.95 before the division.
- **Why not plain sums and weighted means everywhere.** Chosen on the tuning half of the validation subjects (scenario A, step 10 500) and checked on the other half. A voxel of a cold structure next to a hot one is sometimes predicted with the kinetics of its neighbour: with the first version (sums and A0-weighted mean λ_bio) the rest of body had 35% of its predicted A0 in the 10 mm shells around kidneys, glands and bladder, which hold 1.5% of its voxels and of its true A0, and log A0 and log λ_bio were biased by +0.99 and +0.76 while its voxels were calibrated. Hence, for the rest of body only, the predicted voxel quantities next to another source or within 10 mm of kidneys, glands and bladder (where the source definition itself puts the median) are replaced by their median over the other voxels, and λ_bio, constant inside a structure by construction, is read as a median. All of this uses the prediction and the masks only.
- **Lesion rule (README).** The 4 mm labels lose part of a small lesion to its neighbours, so a lesion is summarised on its voxels dilated by one voxel (the ball of `net_data.lesion_dilation`), inside the body. The ring also holds other sources, so the truth is summarised from the rebuilt target maps on the same mask with the same sums, and those values replace the lesion rows of the truth: like with like. The baselines are scored against the cohort value of the lesion; the network is therefore also scored against it in two extra rows (`network, lesions against the cohort value`: the dilated sum, biased high by the ring; and, for the TIAC, the value separated by `net_data.separate_lesion`).
- **Sanity check.** For organs the summary of the rebuilt maps must equal `params.parquet` to float32 rounding (`structure_organ_check`).
- **Uncertainty.** 64 samples of every head from the voxel Gaussians, structure quantities per sample, SD of the (log) quantity over the samples; the mean comes from the voxel means, not from the samples. One standard normal per (structure, head, sample) is shared by all the voxels of the structure: the truth of a structure is one parameter per head painted on its voxels, so the voxel errors of the network are almost fully correlated inside it, and treating them as independent would shrink the SD by about the square root of the number of voxels. The independent-voxel variant is computed only to show this. Heads are sampled independently of each other.

#### 9.3 Structure-level calibration, cross-validated
One SD factor per quantity and structure group (each compartment; all lesions together), the closed-form NLL-optimal one, sqrt(mean z²). Validation is the only data it can be fitted on and also where it is assessed, so the calibrated network in the tables is scaled out of fold: 5 folds grouped by subject, the scales fitted on 4 and applied to the 5th. The scales fitted on all the validation phantoms are saved in `calibration_<scenario>.csv`: this is the file the test evaluation applies, unchanged.

#### 9.4 Comparison with the baselines
`net_data.evaluate(..., by=("structure",))` on the validation virtual patients, main arm, this scenario: n, bias, median absolute error (also as a factor, exp(median |log error|) − 1), RMSE, coverage of the 68% and 95% intervals, and skill = 1 − MAE / MAE of the population baseline. The PBPK-fit baseline is the other 9 draws of the same phantom: it knows the posterior of that phantom, which the network has to infer from the image, so it is a reference for what the PET can give, not a competitor trained on the same information.

Expected in scenario A: positive skill on `log_a0` and `log_tiac`; skill near 0 on `log_lambda_bio`. The PBPK reference itself has some skill on λ_bio where it follows the physiology of the phantom rather than the release rate (on validation: bladder, blood pool, spine, rest of body), so the alarm of `structure_washout` is raised, per structure, when the network skill exceeds max(PBPK skill, 0) + 0.05: that would mean information on the washout reaching the network by a route other than the PET (leakage), and is printed explicitly. The table of the two skills and their difference is always printed.

In a B scenario the washout is correlated with the PET uptake of the draw (ρ in the scenario name), so the network should have positive skill on `log_lambda_bio` for kidneys, liver, spleen, glands, GI and lesions. `structure_washout` then also gives: `skill_ceiling`, the skill of the other 9 draws of the same phantom in the same scenario (`baseline_same_phantom_draws_<scenario>.parquet`), i.e. what a predictor that knows the phantom perfectly can say about a new draw, since the network sees the real PET of the phantom and not the simulated uptake of the draw; `skill_gaussian_bound` = 1 − sqrt(1 − ρ²) (0.20 for ρ = 0.6), the bound if the uptake of the draw itself were known and the washout Gaussian; `share_of_ceiling`. The ceiling is estimated from 9 draws on 27 phantoms: it is noisy (on the training phantoms it is 0.07-0.17 for those structures) and slightly below the true one. The alarm is a network skill above the ceiling + 0.05. The population baseline and the voxel prior are those of the scenario (the marginals of λ_bio are those of A, the spread of the TIAC is not); the PBPK-fit baseline is the same as in A, the PET-fit posterior without the PET-washout link.

Two side counts are printed and saved with the rest: `structure_lesion_overlap` (how many lesions have a dilated mask that holds voxels of another lesion, where the summary mixes the two) and `structure_early_share_cap` (per structure group, the share of voxels whose predicted early share is above the cap of 0.95).

In [ ]:
if RUN_EVALUATION:
    baselines = {"population": pd.read_parquet(scenario_file(NETDATA, "baseline_population", SCENARIO)),
                 "pbpk_fit": pd.read_parquet(f"{NETDATA}/baseline_pbpk_fit.parquet")}
    if SCENARIO != "A":
        baselines[CEILING] = pd.read_parquet(scenario_file(NETDATA, f"baseline_{CEILING}", SCENARIO))
    report = structure_report(nd, results, baselines, val_cohort.phantoms, EVAL_DIR, SCENARIO, EVAL_STAMP,
                              folds=EVAL_FOLDS, seed=EVAL_SEED)
    comparison = report["comparison"]
    print("Organ summaries of the rebuilt maps against params.parquet:\n" + report["organ_check"].to_string(index=False))
    print("\nSD factors fitted on all the validation phantoms (calibration file):\n"
          + report["calibration"].pivot(index="structure", columns="quantity", values="scale").round(2).to_string())
    for column in ("coverage_68", "coverage_95"):
        print(f"\n{column}: correlated, independent voxels, calibrated out of fold\n"
              + side_by_side(comparison, column, (RAW, INDEPENDENT, CALIBRATED)).round(3).to_string())
    for column in ("median_abs_error", "skill"):
        print(f"\n{column}\n" + side_by_side(comparison, column).round(3).to_string())
    washout = report["washout"]
    overlap = report["lesion_overlap"].iloc[0]
    print(f"\nLesions whose dilated mask holds voxels of another lesion: {overlap.with_other_lesion} of {overlap.lesions} "
          f"({overlap.share:.1%})")
    print("\nVoxels with the mean early share above the cap of 0.95:\n" + report["early_share_cap"].round(4).to_string(index=False))
    print("\nSkill on log_lambda_bio, network against the PBPK reference" + (" and the ceiling of the scenario" if SCENARIO != "A" else "")
          + ":\n" + washout.round(3).to_string(index=False))
    if washout.alarm.any():
        print(f"\n[LEAKAGE ALARM] the network predicts the washout better than expected in scenario {SCENARIO} for: "
              f"{', '.join(washout.structure[washout.alarm])}")
    else:
        print(f"\nNo washout alarm: no structure exceeds " + ("max(PBPK skill, 0)" if SCENARIO == "A" else "max(ceiling, 0)")
              + f" + 0.05 in scenario {SCENARIO}.")
    print(f"\nTables and figures in {EVAL_DIR}")

#### 9.5 Training phantoms, same code (train/validation gap)
The validation errors alone cannot separate memorisation from a limit of the method: a lesion that is badly predicted because it is small is badly predicted in training too, one that was memorised is not. `TRAIN_CHECK_PHANTOMS` training phantoms with lesions (evenly spaced in the list sorted by study id, so the choice does not depend on any result) go through the same pass; the voxel scales and the structure scales are those of validation, applied unchanged. Outputs `runs/eval_<scenario>/train_*`, and the maps of these phantoms in `maps_train/` with `SAVE_VAL_MAPS`. The comparison of interest is the weighted mean z² and the NLL of the lesion voxels, and the lesion errors at structure level, against the validation values of sections 9.1-9.4. The local script `diagnose.py` reads these files together with the validation maps.

In [ ]:
if RUN_EVALUATION and RUN_TRAIN_CHECK:
    train_cohort = GpuCohort(nd, NETDATA, SHARD_DIR, device, scenario=SCENARIO, splits=("train",))
    with_lesions = train_cohort.phantoms[train_cohort.phantoms.n_lesions > 0].sort_values("study_id")
    chosen = with_lesions.index.to_numpy()[np.linspace(0, len(with_lesions) - 1, TRAIN_CHECK_PHANTOMS).round().astype(int)]
    train_results = evaluate_cohort(lambda i: predict_phantom(eval_model, train_cohort, i, VALIDATION_PATCH, 2, 0.25),
                                    train_cohort, chosen, loss_cfg, eval_prior, k_samples=EVAL_SAMPLES, seed=EVAL_SEED,
                                    maps_dir=f"{EVAL_DIR}/maps_train" if SAVE_VAL_MAPS else None)
    head_scales = voxel_calibration[voxel_calibration.structure == "all"].set_index("head").head_scale.to_dict()
    _, train_voxel, _ = voxel_report(train_results, EVAL_DIR, SCENARIO, EVAL_STAMP, prefix="train_", scales=head_scales)
    train_report = structure_report(nd, train_results, baselines, train_cohort.phantoms, EVAL_DIR, SCENARIO, EVAL_STAMP,
                                    prefix="train_", calibration=report["calibration"])
    columns = ["head", "structure", "mean_z", "mean_z2", "coverage_68", "nll"]
    gap = voxel_calibration[columns].merge(train_voxel[columns], on=["head", "structure"], suffixes=("_val", "_train"))
    print("Voxel level, validation against training phantoms:\n"
          + gap[gap.structure.isin(["all", "lesion"])].round(3).to_string(index=False))
    lesion_rows = lambda table: table[(table.method == RAW) & (table.structure == "lesion")].set_index("quantity")[
        ["n", "bias", "median_abs_error", "skill"]]
    print("\nLesions at structure level, validation against training phantoms:\n"
          + lesion_rows(comparison).join(lesion_rows(train_report["comparison"]), lsuffix="_val", rsuffix="_train").round(3).to_string())
    del train_cohort, train_results
    clear_vram()

### 10. Test evaluation (frozen, run once)
Switched off. Run a single time, in the frozen test run driven by `test_run.py` (local), after `test_run.py freeze` and `test_run.py spect-data`; nothing is fitted or chosen here. It is the validation pipeline of section 9 as it is now, applied to the test phantoms:

- the same pass (`evaluate_cohort`: sliding windows `VALIDATION_PATCH`, overlap 0.25, 64 correlated samples per structure, prior predictor from `population_prior.parquet`), with the default aggregation of `lib/structures.py` (sums for A0 and TIA, median λ_bio in a compartment and A0-weighted median on a dilated lesion, rest-of-body shells and boundary voxels replaced by their median);
- the voxel SD factors of `voxel_calibration_A.csv` and the structure SD factors of `calibration_A.csv`, both from validation, applied unchanged;
- the gated prior M0 with the choices and network SD factors of `gated_prior_A.json`, unchanged (`test_gated_prior_A.csv`, scored like `gated_prior_A.csv`);
- the same tables and figures as section 9, prefixed `test_`, and the maps of the test phantoms in `maps_test/` (float16), read locally by M3.

**Guards.** The cell refuses to run if any `test_*` output or `maps_test/` already exists (a crashed run is not resumed: what it wrote has to be inspected and removed by hand). Before opening anything it checks the md5 of every frozen file (`calibration_A.csv`, `voxel_calibration_A.csv`, `gated_prior_A.json`, `tto_choice_A.json`, `population_prior.parquet`, `net_data.py`) against `test_freeze_A.json`, and the checkpoint step and validation NLL of `best_A.pth` against those stamped in `calibration_A.csv`; after copying the test shards it checks every test input against `test_inputs_A.json` (written by `test_run.py spect-data`). All these md5 and the md5 of `best_A.pth` are stamped in every output. The test shards and the `*_test` tables are opened only here and in section 12 (`build_test_cohort` lifts the name guard of `data.py` for the construction of this cohort alone).

**Run.** Upload what `test_run.py checklist` lists, set `RUN_TEST_EVALUATION = True` and `RUN_SPECT_UPDATE_TEST = True` (every other flag `False`, `SCENARIO = "A"`), Run all. Estimated cost on an L4: about 0.2 CU of set-up, 0.6 CU for this section (the validation pass of 27 phantoms took about 0.35 CU; 47 test phantoms), 2.0 CU for section 12: about 2.8 CU in all.

In [ ]:
if RUN_TEST_EVALUATION:
    if SCENARIO != "A":
        raise NotImplementedError("The test baselines and SPECT activities of a B scenario are not built")
    done = frozen.existing_test_outputs(EVAL_DIR) + frozen.existing_test_outputs(EVAL_DIR, "maps_test")
    if done:
        raise RuntimeError(f"The test evaluation has already been run, outputs exist: {done[:3]}")
    TEST_STAMP = {**STAMP, **frozen.check_manifest(EVAL_DIR, NETDATA, SRC_DIR)}
    manifest = json.load(open(f"{EVAL_DIR}/{frozen.MANIFEST}"))
    for name in ("cohort", "model", "val_cohort", "train_cohort"):
        globals().pop(name, None)
    clear_vram()
    test_shards = (copy_files_to_local(f"{NETDATA}/shards", test_file_names(NETDATA), f"{LOCAL_DATA}/shards")
                   if IN_COLAB else f"{NETDATA}/shards")
    TEST_STAMP.update(frozen.check_inputs(EVAL_DIR, NETDATA, test_shards))
    best_path = f"{RUN_DIR}/best_{SCENARIO}.pth"
    eval_model = build_model(device, feature_size=FEATURE_SIZE, load_ssl=False, use_checkpoint=False)
    best_step, best_nll, written_by = load_best(eval_model, best_path, device)
    if best_step != manifest["checkpoint_step"] or abs(best_nll - manifest["checkpoint_val_nll"]) > 1e-9:
        raise RuntimeError(f"best_{SCENARIO}.pth (step {best_step}, NLL {best_nll}) is not the checkpoint the validation choices "
                           f"were made with (step {manifest['checkpoint_step']}, NLL {manifest['checkpoint_val_nll']})")
    EVAL_STAMP = {**STAMP, "checkpoint_step": best_step, "checkpoint_val_nll": best_nll}
    TEST_STAMP.update(checkpoint_step=best_step, checkpoint_val_nll=best_nll, checkpoint_md5=frozen.md5(best_path))
    test_cohort = build_test_cohort(GpuCohort, nd, NETDATA, test_shards, device, scenario=SCENARIO)
    test_idx = test_cohort.split_indices("test")
    print(f"{len(test_idx)} test phantoms, {test_cohort.memory_gb():.2f} GB | best_{SCENARIO}.pth step {best_step}, NLL {best_nll:.4f} | "
          f"maps {maps_size_gb(test_cohort, test_idx):.2f} GB in {EVAL_DIR}/maps_test\n" + "\n".join(f"{k}: {v}" for k, v in TEST_STAMP.items()))

    t0 = time.perf_counter()
    test_results = evaluate_cohort(lambda i: predict_phantom(eval_model, test_cohort, i, VALIDATION_PATCH, 2, 0.25),
                                   test_cohort, test_idx, loss_cfg, eval_prior, k_samples=EVAL_SAMPLES, seed=EVAL_SEED,
                                   maps_dir=f"{EVAL_DIR}/maps_test")
    print(f"Evaluation pass: {(time.perf_counter() - t0) / 60:.1f} min")
    head_scales = pd.read_csv(f"{EVAL_DIR}/voxel_calibration_{SCENARIO}.csv")
    head_scales = head_scales[head_scales.structure == "all"].set_index("head").head_scale.to_dict()
    test_nll, test_voxel, test_check = voxel_report(test_results, EVAL_DIR, SCENARIO, TEST_STAMP, prefix="test_", scales=head_scales)
    test_baselines = {"population": pd.read_parquet(f"{NETDATA}/baseline_population_test.parquet"),
                      "pbpk_fit": pd.read_parquet(f"{NETDATA}/baseline_pbpk_fit_test.parquet")}
    test_report = structure_report(nd, test_results, test_baselines, test_cohort.phantoms, EVAL_DIR, SCENARIO,
                                   TEST_STAMP, prefix="test_", calibration=pd.read_csv(f"{EVAL_DIR}/calibration_{SCENARIO}.csv"))
    gate = json.load(open(f"{EVAL_DIR}/gated_prior_{SCENARIO}.json"))
    population = test_baselines["population"]
    test_gated = apply_choice(nd, gate, with_sd(test_results["prediction"]), test_results["truth_like"],
                              population[population.arm == "main"], test_results["truth"])
    save_table(test_gated, f"{EVAL_DIR}/test_gated_prior_{SCENARIO}.csv", TEST_STAMP)

    print("kinetic_loss against the sums of the report:\n" + test_check.round(4).to_string(index=False))
    print("\nNetwork against prior, per head:\n" + test_nll[test_nll.structure == "all"].drop(columns="structure").round(3).to_string(index=False))
    print("\nNetwork minus prior NLL by structure class:\n"
          + test_nll[test_nll.structure != "all"].pivot(index="structure", columns="head", values="network_minus_prior")[list(HEADS)].round(3).to_string())
    shown = ["head", "structure", "mean_z", "mean_z2", "coverage_68", "coverage_95", "nll", "nll_scaled", "coverage_68_scaled", "coverage_95_scaled"]
    print("\nCalibration (voxel SD factors of validation applied):\n" + test_voxel[shown].round(3).to_string(index=False))
    print("\nOrgan summaries of the rebuilt maps against params_test:\n" + test_report["organ_check"].to_string(index=False))
    for column in ("coverage_68", "coverage_95"):
        print(f"\n{column}: correlated, independent voxels, calibrated with the validation factors\n"
              + side_by_side(test_report["comparison"], column, (RAW, INDEPENDENT, CALIBRATED)).round(3).to_string())
    for column in ("median_abs_error", "skill"):
        print(f"\n{column}\n" + side_by_side(test_report["comparison"], column).round(3).to_string())
    overlap = test_report["lesion_overlap"].iloc[0]
    print(f"\nLesions whose dilated mask holds voxels of another lesion: {overlap.with_other_lesion} of {overlap.lesions} ({overlap.share:.1%})")
    print("\nVoxels with the mean early share above the cap of 0.95:\n" + test_report["early_share_cap"].round(4).to_string(index=False))
    washout = test_report["washout"]
    print("\nSkill on log_lambda_bio, network against the PBPK reference:\n" + washout.round(3).to_string(index=False))
    if washout.alarm.any():
        print(f"\n[LEAKAGE ALARM] for: {', '.join(washout.structure[washout.alarm])}")
    print("\nGated prior M0 (choices of gated_prior_A.json):\n" + test_gated.pivot_table(
        index=["structure", "quantity"], columns="method", values="median_abs_error").round(3).to_string())
    print(f"\nTest tables and figures in {EVAL_DIR} (prefix test_), maps in {EVAL_DIR}/maps_test")

### 11. Single-SPECT update: weight-space test-time optimisation (M4)
One post-therapy SPECT of a virtual patient refines the network prediction for that patient. The SPECT is simulated on the 4 mm grid (`lib/spect.py`): true source activities at the SPECT time (`netdata/spect_activity.parquet`, re-simulated from the frozen cohort) × the unit weight of each voxel; the random numbers are fixed by the virtual patient and the time. The simulation and the model that the update methods assume differ on purpose, as an acquisition differs from its model:

- **Resolution**: Gaussian, FWHM drawn per virtual patient between 9 and 12 mm; the methods assume 10 mm.
- **Misregistration**: the image is shifted by a vector of random direction and length between 0 and 2 voxels (0-8 mm); the methods assume none.
- **Calibration**: one factor per image, log-normal with SD 5%, known to the methods only as a distribution.
- **Noise**: Gaussian and spatially correlated, as white noise blurred with a 10 mm FWHM Gaussian; voxel SD = 0.30 × A × (A / 8.35e-7)^−0.17, A the blurred voxel activity as a fraction of the injected activity. Both the amplitude law and the correlation come from 12 organ VOIs of the Deep Blue SPECTs at day 4 and later (local voxel CV 0.24-0.40, log-log slope −0.17 against the activity, autocorrelation length 6.0 mm). On a validation phantom at 96 h this gives a voxel CV of 0.36 in the liver and 0.33 in the kidneys and a CV of the VOI mean of 1.5% and 2.8% (Deep Blue: 0.30 and 0.24-0.32; 1.5% and 3.3-4.6%); independent count-like noise gave 0.25% and 0.44% on the VOI mean.
- **Biology outside the model**: the true activity of every source is the cohort curve with two independent log-normal factors, on the uptake amplitude and on the late biological washout rate, SD 0.15 each (`update.mismatch`), and the truth every method is scored against is perturbed accordingly. The shared parameters of the PBPK cannot represent this, so M5 does not fit the data with the model that generated them. 0.15 is the smallest between-patient log SD of a release rate in the calibrated cohort population (kidneys 0.18, glands 0.15; `nb19_cohort_population_v2.pkl`, notebook 19, itself set from published half-life spreads): the variability of one organ that the model does not carry is taken as large as the smallest variability that it does carry. It is an assumption, not a measurement; the final table is repeated at 0.10 and 0.20 with the same random numbers and the choices made at 0.15.
- **Shape mismatch** (a separate condition, M2 and M5 only): on top of the two factors the late biological washout rate changes in time, λ_bio e^b exp(s (t − 96 h) / 96 h), s normal with SD 0.3 per source (`update.SHAPE_DRIFT_SD`, an assumption without a source: the rate at 24 h and at 168 h differ by a factor 1.6 at one SD). The late phase is then not one exponential, so neither the kinetic model of M2 nor the PBPK contains the truth; the true late TIA is the numerical integral of the curve.

The methods that work on VOI activities (M2, M5) carry a relative model-discrepancy term in their likelihood, one value per structure group, chosen on the tuning half so that the intervals of M2 cover at their nominal rate, then fixed (`tto_choice_<scenario>.json`). Every method is scored against the same truth, the (perturbed) cohort value of the structure, lesions included.

**M4.** A copy of the network is adapted to the one SPECT. Only the output convolution and the last decoder block are trained (`head`): the features that enter that block do not change, so they are computed once per sliding window and every step runs the last block only. The maps of log A0 and log λ_bio start from the gated prior M0: in every structure a constant is added to the network map so that its structure value equals the M0 mean (where M0 is the population, the network keeps only its spatial pattern), and the comparison with M2 and M5 is fair; the M0 rows of the same cases are written next to the M4 rows. The estimate of a structure at a step is the M0 mean plus the change of the structure value of the maps since step 0, so step 0 is M0 exactly, also for the TIAC where M0 is the population (in the second tuning round the TIAC was the late TIA of the maps over 1 − early share of M0, which at step 0 is not the M0 TIAC: 0.314 against 0.263). The maps go through the differentiable kinetics layer (`lib/kinetics.py`: A(t) = A0 exp(−(λ_bio + λ_phys) t)) and the forward model. Two likelihoods: `voxel`, the Gaussian NLL of the SPECT image divided by the number of voxels over which the noise is correlated, with the calibration factor as a parameter; `voi`, the VOI likelihood of M2 (expected VOI sums of the forward image, the same noise, calibration and discrepancy values of `tto_choice_<scenario>.json`), which the misregistration and the resolution of the patient reach only through the VOI totals: the update is then spatially regularised by the network but not driven voxel by voxel. Anchor: the NLL of the maps under their starting values and calibrated voxel SDs, divided by the correlation volume (`voxel`) or averaged within each structure (`voi`), times `anchor_weight`; plus an L2 penalty on the distance of the trained weights from their start. The change of log TIAC is the change of the log late TIA of the maps (the early share stays that of M0).

First tuning round (2026-11-15, voxel likelihood, start from the raw network, anchor not divided by the correlation volume): `norm` at 1e-4 did not move (0.374 → 0.369 over 15 steps), `head` at 1e-4 moved erratically (best at step 9, 0.34 / 0.33 at 48 / 96 h) and did not correct the biases. Second and last round (`weights_rows_tuning2_A.csv`): no setting improved on its own step 0 (0.314) or on M0 (0.263) in median |log TIAC error|; the voxel likelihood gave 0.34-0.95 depending on the learning rate, the VOI likelihood diverged (1.2-5.0). **M4 is frozen as a negative result.** The least bad setting (`head`, voxel likelihood, lr 1e-3, anchor weight 1, 30 steps) is stored in `tto_choice_<scenario>.json` and run once on the reporting half, so that the negative result is reported on the same patients as the other methods. The timing column of the tuning tables was wrong (the first timed run included the warm-up); the cell now times after a warm-up run and stores the measured time of every run.

**Stages and budget.** `SPECT_STAGE = "tuning"`: a few phantoms of the tuning half of the validation subjects (the split stored in `gated_prior_<scenario>.json`), the variants of `SPECT_VARIANTS` (`head` with three learning rates and two anchor weights for the voxel likelihood, three learning rates for the VOI likelihood), predictions at steps 0, 5, 10, 20, 30; the choice (variant and number of steps) is made locally from that table and stored in `tto_choice_<scenario>.json`. `SPECT_STAGE = "reporting"`: the frozen setting on the reporting half (draw 0 of every phantom, 24, 48, 96, 168 h), nothing chosen there. Before running, the cell times a short run, prints the real cost per step and per case and the estimated GPU time and compute units and stops if they exceed `SPECT_CU_LIMIT`. M0 (gated prior), M1 (classical single-time-point), M2 (Bayesian update), M3 (optimisation of the maps) and M5 (differentiable PBPK) run locally on CPU; the comparison M2 / M4 / M5 on the reporting half is the main result.

Expected: the SPECT improves the TIAC and λ_bio substantially over the prior on the structures whose activity follows the late exponential; the gain depends on the SPECT time; M4 should match M2 on organs and can only add something on lesions and on heterogeneity inside a structure.

In [ ]:
if RUN_SPECT_UPDATE:
    import json
    for name in ("cohort", "model"):
        globals().pop(name, None)
    clear_vram()
    SPECT_DIR = f"{EVAL_DIR}/spect"
    os.makedirs(SPECT_DIR, exist_ok=True)
    if "val_cohort" not in globals():
        val_cohort = GpuCohort(nd, NETDATA, SHARD_DIR, device, scenario=SCENARIO, splits=("val",))
    if "eval_model" not in globals():
        eval_model = build_model(device, feature_size=FEATURE_SIZE, load_ssl=False, use_checkpoint=False)
        best_step, best_nll, _ = load_best(eval_model, f"{RUN_DIR}/best_{SCENARIO}.pth", device)
        EVAL_STAMP = {**STAMP, "checkpoint_step": best_step, "checkpoint_val_nll": best_nll}
    gate = json.load(open(f"{EVAL_DIR}/gated_prior_{SCENARIO}.json"))
    tto_choice = json.load(open(f"{EVAL_DIR}/tto_choice_{SCENARIO}.json"))
    half = val_cohort.phantoms.subject.astype(str).map(gate["subject_half"])
    head_scales = pd.read_csv(f"{EVAL_DIR}/voxel_calibration_{SCENARIO}.csv")
    head_scales = head_scales[head_scales.structure == "all"].set_index("head").head_scale.to_dict()
    population = pd.read_parquet(f"{NETDATA}/baseline_population.parquet")
    m0 = gated_table(gate, with_sd(pd.read_csv(f"{EVAL_DIR}/structure_prediction_{SCENARIO}.csv")), population[population.arm == "main"])
    spect_cfg = SpectConfig()

    if SPECT_STAGE == "tuning":
        chosen = val_cohort.phantoms[half == "tuning"].sort_values("n_lesions").index.to_numpy()[::2]
        SPECT_VARIANTS = [dict(subset="head", lr=lr, anchor_weight=w, likelihood="voxel") for lr in (1e-3, 3e-3, 1e-2) for w in (0.1, 1.0)]
        SPECT_VARIANTS += [dict(subset="head", lr=lr, anchor_weight=1.0, likelihood="voi") for lr in (1e-3, 3e-3, 1e-2)]
        SPECT_STEPS, SPECT_TIMES, SPECT_DRAWS = 30, (48.0, 96.0), (0,)
    else:
        chosen = val_cohort.phantoms[half == "reporting"].index.to_numpy()
        SPECT_VARIANTS, SPECT_STEPS = [tto_choice["M4"]["variant"]], int(tto_choice["M4"]["steps"])
        SPECT_TIMES, SPECT_DRAWS = tuple(tto_choice["M4"]["times"]), tuple(tto_choice["M4"]["draws"])
    SPECT_RECORD = tuple(s for s in (0, 5, 10, 20, 30) if s <= SPECT_STEPS) + ((SPECT_STEPS,) if SPECT_STEPS not in (0, 5, 10, 20, 30) else ())
    cases = [(int(i), f"{val_cohort.phantoms.study_id[i]}_d{d:02d}", t) for i in chosen for d in SPECT_DRAWS for t in SPECT_TIMES]
    truth, activity = case_data(NETDATA, list(val_cohort.phantoms.study_id[chosen]), SCENARIO, "mismatch", sd=tto_choice["mismatch_sd"])
    run = lambda some_cases, variants, steps, record, log=print: weight_update_rows(
        eval_model, val_cohort, some_cases, activity, truth, m0, spect_cfg, VALIDATION_PATCH, variants, steps, record,
        tto_choice["discrepancy"], head_scales, loss_cfg.logvar_range, log=log)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    run(cases[:1], SPECT_VARIANTS[:1], 1, (), log=lambda s: None)
    timing = []
    for short in (2, 6):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        run(cases[:1], SPECT_VARIANTS[:1], short, (), log=lambda s: None)
        if device.type == "cuda":
            torch.cuda.synchronize()
        timing.append(time.perf_counter() - t0)
    per_step = max(timing[1] - timing[0], 0.0) / 4
    per_run = max(timing[0] - 2 * per_step, 0.0) + SPECT_STEPS * per_step
    hours = per_run * len(cases) * len(SPECT_VARIANTS) / 3600
    peak = torch.cuda.max_memory_allocated() / 1024**3 if device.type == "cuda" else 0.0
    print(f"{len(cases)} cases x {len(SPECT_VARIANTS)} variants x {SPECT_STEPS} steps | {per_step:.2f} s per step, {per_run:.0f} s per run "
          f"(first case, after a warm-up run), peak memory {peak:.1f} GB | estimate {hours:.2f} h" + (f" = {hours * CU_PER_HOUR:.1f} CU" if CU_PER_HOUR else ""))
    if CU_PER_HOUR and hours * CU_PER_HOUR > SPECT_CU_LIMIT:
        raise RuntimeError(f"Estimated cost above SPECT_CU_LIMIT = {SPECT_CU_LIMIT} CU: reduce the cases, the variants or the steps")

    t0 = time.perf_counter()
    weights_table = pd.DataFrame(run(cases, SPECT_VARIANTS, SPECT_STEPS, SPECT_RECORD))
    measured_hours = (time.perf_counter() - t0) / 3600
    runs = weights_table.dropna(subset=["seconds"]).drop_duplicates(["virtual_id", "t_h", "method"]).seconds
    print(f"Measured: {measured_hours:.2f} h" + (f" = {measured_hours * CU_PER_HOUR:.1f} CU" if CU_PER_HOUR else "")
          + f" | {runs.median():.1f} s per run (median), {runs.median() / SPECT_STEPS:.2f} s per step")
    save_table(weights_table, f"{SPECT_DIR}/weights_rows_{SPECT_TAG if SPECT_STAGE == 'tuning' else SPECT_STAGE}_{SCENARIO}.csv",
               {**EVAL_STAMP, "seconds_per_step": float(runs.median() / SPECT_STEPS), "seconds_per_run": float(runs.median()),
                "hours_total": measured_hours})
    weights_table["error"] = (weights_table["mean"] - weights_table.truth).abs()
    structures = ["kidneys_functional", "liver", "spleen", "salivary_glands", "gi_tract", "lesion"]
    for q in ("log_tiac", "log_a0", "log_lambda_bio"):
        d = weights_table[(weights_table.quantity == q) & weights_table.structure.isin(structures)]
        medians = d.groupby(["method", "t_h", "step", "structure"]).error.median().groupby(["method", "t_h", "step"]).mean()
        print(f"\n{q}: median |error|, mean over kidneys, liver, spleen, glands, GI, lesions\n" + medians.unstack(["t_h", "step"]).round(3).to_string())
    best = weights_table[(weights_table.quantity == "log_tiac") & (weights_table.step.isin([0, SPECT_STEPS]))]
    print("\nlog_tiac by structure, M0 and the last step of every variant\n" + best.pivot_table(
        index="structure", columns=["method", "t_h"], values="error", aggfunc="median").round(2).T.to_string())


### 12. Single-SPECT update on the test split, M4 (frozen, run once)
Switched off. Run in the same session as section 10 (it reads `test_structure_prediction_A.csv` and reuses the test cohort and the model if they are in memory; otherwise it rebuilds them with the same checks). M4 with the setting stored in `tto_choice_A.json["M4"]` (`head`, voxel likelihood, lr 1e-3, anchor weight 1, 30 steps), at the four SPECT times and draw 0 of every test phantom, main condition (mismatch SD from the same file), steps recorded at 0, 5, 10, 20, 30 as on the reporting half; the discrepancy values, the gated prior M0 (`gated_prior_A.json` on the test predictions and `baseline_population_test`) and the voxel SD factors are read from the frozen files, nothing is chosen. The SPECT activities are `netdata/spect_activity_test.parquet`, written locally by `test_run.py spect-data` (`spect_data.build_test`): same generator and seeds as validation, derived from the test virtual ids and the SPECT time; truth and activities pass through `update.case_data(..., test=True)`, the function of the local methods, so M4 and M0-M3, M5 see the same images.

Cost: the measured time per run on the reporting half (`weights_rows_reporting_A.csv`: 60 runs in 0.33 h, about 20 s each) times the number of runs is printed before starting; the cell stops if it exceeds `TEST_CU_LIMIT`. It refuses to run if `spect/test_weights_rows_A.csv` exists. Output stamped with the md5 of the frozen files, of the test inputs and of the checkpoint.

In [ ]:
if RUN_SPECT_UPDATE_TEST:
    if SCENARIO != "A":
        raise NotImplementedError("The test baselines and SPECT activities of a B scenario are not built")
    SPECT_DIR = f"{EVAL_DIR}/spect"
    test_out = f"{SPECT_DIR}/test_weights_rows_{SCENARIO}.csv"
    frozen.refuse_if_done([test_out])
    prediction_path = f"{EVAL_DIR}/test_structure_prediction_{SCENARIO}.csv"
    if not os.path.exists(prediction_path):
        raise FileNotFoundError("test_structure_prediction_A.csv not found: run section 10 first")
    SPECT_TEST_STAMP = {**STAMP, **frozen.check_manifest(EVAL_DIR, NETDATA, SRC_DIR)}
    manifest = json.load(open(f"{EVAL_DIR}/{frozen.MANIFEST}"))
    os.makedirs(SPECT_DIR, exist_ok=True)
    for name in ("cohort", "model", "val_cohort", "train_cohort"):
        globals().pop(name, None)
    test_shards = (copy_files_to_local(f"{NETDATA}/shards", test_file_names(NETDATA), f"{LOCAL_DATA}/shards")
                   if IN_COLAB else f"{NETDATA}/shards")
    SPECT_TEST_STAMP.update(frozen.check_inputs(EVAL_DIR, NETDATA, test_shards))
    if "test_cohort" not in globals():
        clear_vram()
        test_cohort = build_test_cohort(GpuCohort, nd, NETDATA, test_shards, device, scenario=SCENARIO)
    best_path = f"{RUN_DIR}/best_{SCENARIO}.pth"
    if "TEST_STAMP" in globals() and "eval_model" in globals():
        best_step, best_nll = TEST_STAMP["checkpoint_step"], TEST_STAMP["checkpoint_val_nll"]
    else:
        eval_model = build_model(device, feature_size=FEATURE_SIZE, load_ssl=False, use_checkpoint=False)
        best_step, best_nll, _ = load_best(eval_model, best_path, device)
    if best_step != manifest["checkpoint_step"] or abs(best_nll - manifest["checkpoint_val_nll"]) > 1e-9:
        raise RuntimeError(f"best_{SCENARIO}.pth (step {best_step}) is not the frozen checkpoint (step {manifest['checkpoint_step']})")
    SPECT_TEST_STAMP.update(checkpoint_step=best_step, checkpoint_val_nll=best_nll, checkpoint_md5=frozen.md5(best_path))
    choice_all = json.load(open(f"{EVAL_DIR}/tto_choice_{SCENARIO}.json"))
    m4 = choice_all["M4"]
    steps = int(m4["steps"])
    record = tuple(s for s in (0, 5, 10, 20, 30) if s <= steps) + ((steps,) if steps not in (0, 5, 10, 20, 30) else ())
    test_idx = test_cohort.split_indices("test")
    studies = list(test_cohort.phantoms.study_id[test_idx])
    cases = [(int(i), f"{test_cohort.phantoms.study_id[i]}_d{d:02d}", float(t)) for i in test_idx for d in m4["draws"] for t in m4["times"]]
    truth, activity = case_data(NETDATA, studies, SCENARIO, "mismatch", sd=choice_all["mismatch_sd"], test=True)
    head_scales = pd.read_csv(f"{EVAL_DIR}/voxel_calibration_{SCENARIO}.csv")
    head_scales = head_scales[head_scales.structure == "all"].set_index("head").head_scale.to_dict()
    gate = json.load(open(f"{EVAL_DIR}/gated_prior_{SCENARIO}.json"))
    population = pd.read_parquet(f"{NETDATA}/baseline_population_test.parquet")
    m0 = gated_table(gate, with_sd(pd.read_csv(prediction_path)), population[population.arm == "main"])

    reporting = f"{SPECT_DIR}/weights_rows_reporting_{SCENARIO}.csv"
    if os.path.exists(reporting):
        measured = pd.read_csv(reporting, usecols=["virtual_id", "t_h", "hours_total"])
        seconds_per_run = float(measured.hours_total.iloc[0]) * 3600 / len(measured[["virtual_id", "t_h"]].drop_duplicates())
    else:
        seconds_per_run = float(m4["seconds_per_step"]) * (steps + 1)
    hours = seconds_per_run * len(cases) / 3600
    print(f"{len(cases)} cases ({len(test_idx)} phantoms x draws {m4['draws']} x {m4['times']} h), setting {m4['variant']}, {steps} steps | "
          f"{seconds_per_run:.0f} s per run on the reporting half -> estimate {hours:.2f} h" + (f" = {hours * CU_PER_HOUR:.1f} CU" if CU_PER_HOUR else ""))
    if CU_PER_HOUR and hours * CU_PER_HOUR > TEST_CU_LIMIT:
        raise RuntimeError(f"Estimated cost above TEST_CU_LIMIT = {TEST_CU_LIMIT} CU")
    t0 = time.perf_counter()
    rows = pd.DataFrame(weight_update_rows(eval_model, test_cohort, cases, activity, truth, m0, SpectConfig(), VALIDATION_PATCH,
                                           [m4["variant"]], steps, record, choice_all["discrepancy"], head_scales, loss_cfg.logvar_range))
    measured_hours = (time.perf_counter() - t0) / 3600
    save_table(rows, test_out, {**SPECT_TEST_STAMP, "hours_total": measured_hours})
    print(f"Measured: {measured_hours:.2f} h" + (f" = {measured_hours * CU_PER_HOUR:.1f} CU" if CU_PER_HOUR else "") + f" | written {test_out}")
    rows["error"] = (rows["mean"] - rows.truth).abs()
    uptake = ["kidneys_functional", "liver", "spleen", "salivary_glands", "gi_tract", "lesion"]
    d = rows[(rows.quantity == "log_tiac") & rows.structure.isin(uptake) & rows.step.isin([0, steps])]
    print("\nlog_tiac, median |error|, mean over kidneys, liver, spleen, glands, GI, lesions\n"
          + d.groupby(["method", "t_h", "step", "structure"]).error.median().groupby(["method", "t_h", "step"]).mean().unstack("t_h").round(3).to_string())

### 13. Stop the runtime
The tables and checkpoints are already on Drive. Drive is flushed and the runtime released, so no compute units are spent after the last cell.

In [ ]:
if STOP_RUNTIME_AT_END and IN_COLAB:
    stop_runtime()